# Notebook 54: STRAT-004 Forward Test (Past Year)

**Goal:** Validate STRAT-004 performance on most recent data (2024-2025).

This is critical because:
- Strategy was developed on historical data
- Need to confirm it works in current market conditions
- Recent year is "out of sample" for mental model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
from pathlib import Path
from datetime import datetime, timedelta

DATA_DIR = Path("../data")
HOURLY_DIR = DATA_DIR / "hourly"

# Define forward test period
FORWARD_START = "2024-01-15"  # 1 year ago from today
FORWARD_END = "2025-01-15"    # Today

print(f"Forward Test Period: {FORWARD_START} to {FORWARD_END}")

## 1. Load Data

In [ ]:
# Load hourly data
def load_hourly():
    price = pd.read_parquet(HOURLY_DIR / "price.parquet")
    sopr = pd.read_parquet(HOURLY_DIR / "sopr.parquet")
    sopr_sth = pd.read_parquet(HOURLY_DIR / "sopr_sth.parquet")
    realized_loss = pd.read_parquet(HOURLY_DIR / "realized_loss.parquet")
    
    df = price.rename(columns={"value": "price"}).set_index("time")
    df["sopr"] = sopr.set_index("time")["value"]
    df["sopr_sth"] = sopr_sth.set_index("time")["value"]
    df["realized_loss"] = realized_loss.set_index("time")["value"]
    return df

df_full = load_hourly()
print(f"Full dataset: {len(df_full):,} hourly bars")
print(f"Date range: {df_full.index.min()} to {df_full.index.max()}")

In [ ]:
# Calculate z-score using FULL history (not just forward period)
# This simulates real-world usage where we have historical data

def add_zscore(df, window):
    df = df.copy()
    df["rl_mean"] = df["realized_loss"].rolling(window=window, min_periods=window//2).mean()
    df["rl_std"] = df["realized_loss"].rolling(window=window, min_periods=window//2).std()
    df["rl_zscore"] = (df["realized_loss"] - df["rl_mean"]) / df["rl_std"]
    return df

# Use 1 year lookback for z-score
df_full = add_zscore(df_full, 365 * 24)  # 8760 hours

# Now filter to forward test period
df_forward = df_full[(df_full.index >= FORWARD_START) & (df_full.index <= FORWARD_END)].dropna()

print(f"\nForward test data: {len(df_forward):,} hourly bars")
print(f"Price range: ${df_forward['price'].min():,.0f} to ${df_forward['price'].max():,.0f}")

## 2. Run STRAT-004 on Forward Period

In [ ]:
def run_strat004(df, trail=0.12):
    """STRAT-004: 1H 12% trail"""
    cond = (df["sopr"] < 1) & (df["sopr_sth"] < 1) & (df["rl_zscore"] > 0.5)
    entry = cond & ~cond.shift(1).fillna(False)
    
    if entry.sum() == 0:
        return None, entry
    
    pf = vbt.Portfolio.from_signals(
        close=df["price"],
        entries=entry,
        exits=None,
        sl_stop=trail,
        sl_trail=True,
        freq="1h",
        init_cash=10000,
        fees=0.001
    )
    
    return pf, entry

pf, entries = run_strat004(df_forward)

if pf is None:
    print("❌ NO TRADES in forward period!")
else:
    print(f"✅ Strategy executed: {entries.sum()} entry signals")

In [ ]:
# Get metrics
def get_metrics(pf, days):
    trades = pf.trades.records_readable
    if len(trades) == 0:
        return None
    
    years = days / 365.25
    total_return = pf.total_return() * 100
    
    durations = (trades["Exit Timestamp"] - trades["Entry Timestamp"]).dropna()
    avg_days = durations.mean().total_seconds() / 86400 if len(durations) > 0 else 0
    
    winning = trades[trades["PnL"] > 0]
    losing = trades[trades["PnL"] < 0]
    
    return {
        "return": total_return,
        "cagr": ((1 + total_return/100) ** (1/years) - 1) * 100 if years > 0 else 0,
        "sharpe": pf.sharpe_ratio(),
        "max_dd": pf.max_drawdown() * 100,
        "trades": len(trades),
        "trades_yr": len(trades) / years if years > 0 else 0,
        "avg_days": avg_days,
        "win_rate": (trades["PnL"] > 0).mean() * 100,
        "profit_factor": abs(winning["PnL"].sum() / losing["PnL"].sum()) if len(losing) > 0 and losing["PnL"].sum() != 0 else np.inf,
        "avg_win": winning["Return"].mean() * 100 if len(winning) > 0 else 0,
        "avg_loss": losing["Return"].mean() * 100 if len(losing) > 0 else 0,
        "best_trade": trades["Return"].max() * 100,
        "worst_trade": trades["Return"].min() * 100,
    }

if pf:
    days = (df_forward.index.max() - df_forward.index.min()).days
    m = get_metrics(pf, days)
    
    # Buy & hold comparison
    bh_return = (df_forward["price"].iloc[-1] / df_forward["price"].iloc[0] - 1) * 100

In [ ]:
print("="*80)
print("STRAT-004 FORWARD TEST RESULTS (Past Year)")
print("="*80)

if pf and m:
    print(f"\nPeriod: {FORWARD_START} to {FORWARD_END} ({days} days)")
    print(f"\n{'Metric':<25} {'Value':>15}")
    print("-"*45)
    print(f"{'Total Return':<25} {m['return']:>+14.1f}%")
    print(f"{'Buy & Hold Return':<25} {bh_return:>+14.1f}%")
    print(f"{'Outperformance':<25} {m['return'] - bh_return:>+14.1f}%")
    print("-"*45)
    print(f"{'Annualized (CAGR)':<25} {m['cagr']:>+14.1f}%")
    print(f"{'Sharpe Ratio':<25} {m['sharpe']:>15.2f}")
    print(f"{'Max Drawdown':<25} {m['max_dd']:>14.1f}%")
    print("-"*45)
    print(f"{'Total Trades':<25} {m['trades']:>15}")
    print(f"{'Trades/Year (annualized)':<25} {m['trades_yr']:>15.1f}")
    print(f"{'Avg Hold (days)':<25} {m['avg_days']:>15.1f}")
    print("-"*45)
    print(f"{'Win Rate':<25} {m['win_rate']:>14.0f}%")
    print(f"{'Profit Factor':<25} {m['profit_factor']:>15.2f}")
    print(f"{'Avg Win':<25} {m['avg_win']:>+14.1f}%")
    print(f"{'Avg Loss':<25} {m['avg_loss']:>+14.1f}%")
    print(f"{'Best Trade':<25} {m['best_trade']:>+14.1f}%")
    print(f"{'Worst Trade':<25} {m['worst_trade']:>+14.1f}%")
else:
    print("\n❌ No trades executed in forward period")
    print("   This could mean: no capitulation events occurred")

## 3. Trade Details

In [ ]:
if pf:
    trades = pf.trades.records_readable
    
    print("\n" + "="*100)
    print("INDIVIDUAL TRADES (Forward Period)")
    print("="*100)
    
    # Use correct vectorbt column names
    display_df = trades.copy()
    display_df["Return %"] = display_df["Return"] * 100
    display_df["Days"] = (trades["Exit Timestamp"] - trades["Entry Timestamp"]).dt.total_seconds() / 86400
    display_df["Win/Loss"] = np.where(display_df["PnL"] > 0, "✅ WIN", "❌ LOSS")
    
    print("\n")
    for i, row in display_df.iterrows():
        print(f"Trade {i+1}: {row['Win/Loss']}")
        print(f"  Entry:  {row['Entry Timestamp']} @ ${row['Avg Entry Price']:,.0f}")
        print(f"  Exit:   {row['Exit Timestamp']} @ ${row['Avg Exit Price']:,.0f}")
        print(f"  Hold:   {row['Days']:.1f} days")
        print(f"  Return: {row['Return %']:+.1f}%  (${row['PnL']:+,.0f})")
        print()

## 4. Equity Curve

In [ ]:
if pf:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [2, 1]})
    
    # Top: Equity curves
    ax1 = axes[0]
    
    # STRAT-004 equity
    equity = pf.value().resample('D').last()
    equity.plot(ax=ax1, label=f"STRAT-004: {m['return']:+.1f}%", linewidth=2, color='blue')
    
    # Buy & Hold
    bh = (df_forward["price"].resample('D').last() / df_forward["price"].iloc[0]) * 10000
    bh.plot(ax=ax1, label=f"Buy & Hold: {bh_return:+.1f}%", linewidth=2, color='gray', linestyle='--')
    
    # Mark trades
    for _, trade in trades.iterrows():
        entry_date = trade["Entry Timestamp"]
        exit_date = trade["Exit Timestamp"]
        color = 'green' if trade["PnL"] > 0 else 'red'
        ax1.axvline(x=entry_date, color=color, alpha=0.3, linestyle='-')
        ax1.axvline(x=exit_date, color=color, alpha=0.3, linestyle='--')
    
    ax1.set_title(f"STRAT-004 Forward Test: {FORWARD_START} to {FORWARD_END}")
    ax1.set_ylabel("Portfolio Value ($)")
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Bottom: Price with entry signals
    ax2 = axes[1]
    
    price_daily = df_forward["price"].resample('D').last()
    ax2.plot(price_daily.index, price_daily.values, color='black', alpha=0.7)
    
    # Mark entry/exit points (using correct column names)
    for _, trade in trades.iterrows():
        color = 'green' if trade["PnL"] > 0 else 'red'
        ax2.scatter(trade["Entry Timestamp"], trade["Avg Entry Price"], color=color, marker='^', s=100, zorder=5)
        ax2.scatter(trade["Exit Timestamp"], trade["Avg Exit Price"], color=color, marker='v', s=100, zorder=5)
    
    ax2.set_ylabel("BTC Price ($)")
    ax2.set_xlabel("Date")
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Signal Analysis - When Did Signals Fire?

In [ ]:
# Show when entry conditions were met
print("\n" + "="*80)
print("ENTRY SIGNAL ANALYSIS")
print("="*80)

# Find all times when entry conditions were met
cond = (df_forward["sopr"] < 1) & (df_forward["sopr_sth"] < 1) & (df_forward["rl_zscore"] > 0.5)
signal_bars = df_forward[cond]

print(f"\nTotal bars in period: {len(df_forward):,}")
print(f"Bars with entry signal: {len(signal_bars):,} ({len(signal_bars)/len(df_forward)*100:.2f}%)")

if len(signal_bars) > 0:
    print(f"\nSignal periods:")
    
    # Group consecutive signals
    signal_bars = signal_bars.copy()
    signal_bars["group"] = (signal_bars.index.to_series().diff() > pd.Timedelta(hours=24)).cumsum()
    
    for group_id, group in signal_bars.groupby("group"):
        start = group.index.min()
        end = group.index.max()
        duration = (end - start).total_seconds() / 3600
        avg_price = group["price"].mean()
        print(f"  {start.strftime('%Y-%m-%d %H:%M')} to {end.strftime('%Y-%m-%d %H:%M')} ({duration:.0f}h) @ ${avg_price:,.0f}")
else:
    print("\n⚠️ No entry signals in forward period!")
    print("   This means no capitulation events occurred.")

## 6. Compare to Historical Performance

In [ ]:
# Run on full history for comparison
print("\n" + "="*80)
print("FORWARD VS HISTORICAL COMPARISON")
print("="*80)

# Historical period (2019-2024)
df_hist = df_full[(df_full.index >= "2019-01-01") & (df_full.index < FORWARD_START)].dropna()
pf_hist, _ = run_strat004(df_hist)

if pf_hist:
    hist_days = (df_hist.index.max() - df_hist.index.min()).days
    m_hist = get_metrics(pf_hist, hist_days)
    
    print(f"\n{'Metric':<25} {'Historical':>15} {'Forward':>15} {'Difference':>15}")
    print(f"{'(2019-2024)':<25} {'':>15} {'(Past Year)':<15}")
    print("-"*75)
    
    if m:
        metrics_compare = [
            ("CAGR", "cagr", "%"),
            ("Sharpe", "sharpe", ""),
            ("Max Drawdown", "max_dd", "%"),
            ("Trades/Year", "trades_yr", ""),
            ("Win Rate", "win_rate", "%"),
            ("Avg Hold Days", "avg_days", ""),
        ]
        
        for name, key, suffix in metrics_compare:
            h = m_hist[key]
            f = m[key]
            diff = f - h
            print(f"{name:<25} {h:>14.1f}{suffix} {f:>14.1f}{suffix} {diff:>+14.1f}{suffix}")
    else:
        print("No forward trades to compare")

## 7. Forward Test Verdict

In [ ]:
print("\n" + "="*80)
print("FORWARD TEST VERDICT")
print("="*80)

if pf and m:
    # Determine verdict
    passed_return = m["return"] > 0
    passed_vs_bh = m["return"] > bh_return * 0.5  # At least 50% of B&H
    passed_trades = m["trades"] >= 1
    passed_winrate = m["win_rate"] >= 35
    
    tests_passed = sum([passed_return, passed_vs_bh, passed_trades, passed_winrate])
    
    print(f"\n✓ Positive Return:        {'✅ PASS' if passed_return else '❌ FAIL'} ({m['return']:+.1f}%)")
    print(f"✓ Competitive vs B&H:     {'✅ PASS' if passed_vs_bh else '❌ FAIL'} ({m['return']:+.1f}% vs {bh_return:+.1f}%)")
    print(f"✓ Generated Trades:       {'✅ PASS' if passed_trades else '❌ FAIL'} ({m['trades']} trades)")
    print(f"✓ Win Rate > 35%:         {'✅ PASS' if passed_winrate else '❌ FAIL'} ({m['win_rate']:.0f}%)")
    
    print(f"\n{'='*40}")
    if tests_passed >= 3:
        print(f"VERDICT: ✅ STRAT-004 VALIDATED ({tests_passed}/4 tests passed)")
        print(f"\nStrategy performed as expected in forward test.")
        print(f"Ready for live deployment with proper risk management.")
    elif tests_passed >= 2:
        print(f"VERDICT: ⚠️ PARTIALLY VALIDATED ({tests_passed}/4 tests passed)")
        print(f"\nStrategy showed mixed results. Consider:")
        print(f"- Paper trading for another month")
        print(f"- Smaller position sizes initially")
    else:
        print(f"VERDICT: ❌ NOT VALIDATED ({tests_passed}/4 tests passed)")
        print(f"\nStrategy underperformed in forward test.")
        print(f"Recommend further analysis before live trading.")
else:
    print("\n⚠️ NO TRADES IN FORWARD PERIOD")
    print("\nThis is not necessarily bad - it means:")
    print("1. No capitulation events occurred in the past year")
    print("2. The strategy is selective (which is by design)")
    print("\nCheck if there were any near-misses on entry conditions.")

In [ ]:
# Check how close we came to signals
print("\n" + "="*80)
print("NEAR-MISS ANALYSIS")
print("="*80)

# Check each condition separately
sopr_met = (df_forward["sopr"] < 1).mean() * 100
sth_met = (df_forward["sopr_sth"] < 1).mean() * 100
rl_met = (df_forward["rl_zscore"] > 0.5).mean() * 100
all_met = ((df_forward["sopr"] < 1) & (df_forward["sopr_sth"] < 1) & (df_forward["rl_zscore"] > 0.5)).mean() * 100

print(f"\nCondition met frequency:")
print(f"  SOPR < 1:           {sopr_met:5.1f}% of bars")
print(f"  STH-SOPR < 1:       {sth_met:5.1f}% of bars")
print(f"  RL z-score > 0.5:   {rl_met:5.1f}% of bars")
print(f"  ALL conditions:     {all_met:5.1f}% of bars")

# Find periods that came close (2 of 3 conditions met)
two_of_three = (
    ((df_forward["sopr"] < 1) & (df_forward["sopr_sth"] < 1)) |
    ((df_forward["sopr"] < 1) & (df_forward["rl_zscore"] > 0.5)) |
    ((df_forward["sopr_sth"] < 1) & (df_forward["rl_zscore"] > 0.5))
)
print(f"  2 of 3 conditions:  {two_of_three.mean()*100:5.1f}% of bars")